# Training-mask alignment

Creates a finite training manifest where the supervised assistant target is not covered by the loss mask and proves the packing checker reports the defect.


The code cell below builds a minimal in-memory artifact, runs the real PromptABI checker, and asserts the proof obligation.


In [ ]:
from promptabi import ArtifactKind, ArtifactLocation, LossMaskPolicy, LossMaskStrategy
from promptabi import PackingStrategy, PackingWindow, TrainingDatasetKind, TrainingDatasetSpec
from promptabi import TrainingManifestArtifact, TrainingPackingFindingKind, TrainingSpanContract
from promptabi import analyze_training_packing

manifest = TrainingManifestArtifact(
    kind=ArtifactKind.TRAINING_MANIFEST,
    name="mask-defect",
    location=ArtifactLocation(uri="memory://mask-defect"),
    datasets=(TrainingDatasetSpec(name="sft", kind=TrainingDatasetKind.SUPERVISED, format="chat-jsonl"),),
    loss_mask_policy=LossMaskPolicy(strategy=LossMaskStrategy.ASSISTANT_ONLY, target_roles=("assistant",)),
    packing_window=PackingWindow(
        strategy=PackingStrategy.SAMPLE_PACKING,
        max_tokens=64,
        boundary_token="<eos>",
        preserve_example_boundaries=True,
    ),
    supervised_spans=(
        TrainingSpanContract(
            span_id="row-1.assistant",
            target_role="assistant",
            rendered_region_role="assistant",
            start_token=12,
            end_token=20,
            region_start_token=10,
            region_end_token=22,
            supervised_target=True,
            loss_masked=False,
        ),
    ),
)
report = analyze_training_packing(manifest)
mask_findings = [finding for finding in report.findings if finding.kind is TrainingPackingFindingKind.MASK_DROPPED]

assert mask_findings
assert mask_findings[0].span_id == "row-1.assistant"
assert "loss mask" in mask_findings[0].message
report
